In [1]:
from pathlib import Path
import json
import csv
from collections import defaultdict, Counter
import networkx as nx
import random
from itertools import combinations
from multiprocessing import Pool

import pandas as pd
import numpy as np
from scipy.stats import mode
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from IPython.display import YouTubeVideo, HTML, display

In [2]:
def first_start(timings):
    """Parse start_times like '86' or '12,45,90' → int (first value), or None."""
    s = timings.get("start_times")
    if not s:
        return None
    try:
        return int(s.split(",")[0].strip())
    except (ValueError, AttributeError):
        return None
    
def yt_iframe(video_id, start=0, width=400, height=225):
    return (f'<iframe width="{width}" height="{height}" '
            f'src="https://www.youtube.com/embed/{video_id}?start={start}" '
            f'frameborder="0" allowfullscreen></iframe>')

def print_chain(chain, sub):
    """Print edge details and YouTube iframes for each step in a chain through `sub`."""
    for i in range(len(chain) - 1):
        u, v = chain[i], chain[i + 1]
        edge_id = sub[u][v]["edge_id"]
        e = edges[edge_id]
        src_start = first_start(e["src_timings"]) or 0
        dst_start = first_start(e["dst_timings"]) or 0
        print(f"\nEdge {i}  [{e['part_sampled']}]  edge_id={edge_id}")
        print(f"  src: {e['src_artist']} — {e['src_song']}  ({u})  @ {src_start}s")
        print(f"  dst: {e['dst_artist']} — {e['dst_song']}  ({v})  @ {dst_start}s")
        print(f"  part_sampled: {e['part_sampled']}")
        display(HTML(f"""
        <div style="display:flex; gap:12px; margin:8px 0;">
          <div>{yt_iframe(u, src_start)}</div>
          <div>{yt_iframe(v, dst_start)}</div>
        </div>
        """))

## Directories

In [3]:
dir_samplepairs = Path('/projects/mtg/projects/sample-identification/datasets/samplepairs_withnoise')
dir_sample100 = Path('/projects/mtg/projects/sample-identification/datasets/sample100_withnoise/')

dir_whosampled = Path("/projects/mtg/projects/whosampled")
dir_metadata = dir_whosampled/ "metadata"
dir_url = dir_whosampled/ "url"

dir_wav = Path("/projects/mtg/projects/sample-identification/datasets/whosampled-wav-16khz")

path_whosampled_annotations_all = dir_whosampled / "whosampled_251120_pairs"
path_whosampled_annotations = dir_whosampled / "whosampled_251120_pairs.clean.download_complete"
path_whosampled_url = dir_url / "sample.url"
path_whosampled_url_missing = dir_url / "sample.url.missing"

split_dir = Path("/projects/mtg/projects/sample-identification/datasets/splits-27052026")

In [4]:
paths_metadata = list(dir_metadata.rglob("*.json"))
print(f"{len(paths_metadata):,} metadata files downloaded.")

271,652 metadata files downloaded.


In [5]:
paths_wav = list(dir_wav.rglob("*.wav"))
print(f"{len(paths_wav):,} audio files preprocessed.")

189,857 audio files preprocessed.


In [6]:
with open(path_whosampled_url) as in_f:
    paths_url = [line.strip() for line in in_f if line.strip()]
print(f"{len(paths_url):,} edges exist in WhoSampled.")

whosampled_ids_all = {url.split("sample/")[1].split("/")[0] for url in paths_url}
print(len(whosampled_ids_all))
    
with open(path_whosampled_url_missing) as in_f:
    paths_url_missing = [line.strip() for line in in_f if line.strip()]
print(f"{len(paths_url_missing):,} edges could not be downloaded.")

whosampled_ids_missing = {url.split("sample/")[1].split("/")[0] for url in paths_url_missing}
print(len(whosampled_ids_missing))

547,511 edges exist in WhoSampled.
546725
281,498 edges could not be downloaded.
281498


## Load the Annotations

In [7]:
with open(path_whosampled_annotations_all) as in_f:
    whosampled_annotations_all = [json.loads(line) for line in in_f]
print(f"{len(whosampled_annotations_all):,} annotations mined.")

with open(path_whosampled_annotations) as in_f:
    whosampled_annotations = [json.loads(line) for line in in_f]
print(f"{len(whosampled_annotations):,} annotations available.")

272,174 annotations mined.
180,801 annotations available.


In [8]:
whosampled_annotations[0]

{'url': 'https://www.whosampled.com/sample/202869/Bomfunk-MC%27s-Max%27C-Live-Yor-Live-The-Winstons-Amen,-Brother/',
 'dst_song': 'Live Yor Live',
 'dst_artist': ["Bomfunk MC's", "Max'C"],
 'dst_album': "Burnin' Sneakers",
 'dst_audio_id': 'QDCzmc4IVMQ',
 'dst_provider': 'youtube',
 'src_song': 'Amen, Brother',
 'src_artist': ['The Winstons'],
 'src_album': 'Color Him Father',
 'src_audio_id': 'GxZuq57_bYM',
 'src_provider': 'youtube',
 'dst_timings': {'start_times': '78', 'throughout': True},
 'src_timings': {'start_times': '86', 'throughout': False}}

## Prepare the Annotations

In [9]:
# Collect information
edges = {}
for i, row in enumerate(whosampled_annotations):
    row = row.copy()  # shallow copy of the dict
    for k in ["dst_album", "dst_provider", "src_album", "src_provider"]:
        row.pop(k)
    _id = row["url"].split("https://www.whosampled.com/sample/")[1].split("/")[0]
    # NOTE: whosampled id collisions happen... better to use our ids
    edges[_id] = row
print(f"{len(edges):,} edges found.")
ids_whosampled_success = set(edges.keys())

# Remove the metadata that we couldn't download
paths_metadata = [p for p in paths_metadata if p.stem in ids_whosampled_success]
print(f"{len(paths_metadata):,}")

180,500 edges found.
180,500


In [10]:
edges[list(edges.keys())[0]]

{'url': 'https://www.whosampled.com/sample/202869/Bomfunk-MC%27s-Max%27C-Live-Yor-Live-The-Winstons-Amen,-Brother/',
 'dst_song': 'Live Yor Live',
 'dst_artist': ["Bomfunk MC's", "Max'C"],
 'dst_audio_id': 'QDCzmc4IVMQ',
 'src_song': 'Amen, Brother',
 'src_artist': ['The Winstons'],
 'src_audio_id': 'GxZuq57_bYM',
 'dst_timings': {'start_times': '78', 'throughout': True},
 'src_timings': {'start_times': '86', 'throughout': False}}

In [11]:
# Add part-sampled information from metadata to the edges and count
counts_part_sampled = Counter()
for path in paths_metadata:
    with open(path) as f_in:
        metadata = json.load(f_in)
    counts_part_sampled[metadata["part_sampled"]] += 1
    edges[path.stem]["part_sampled"] = metadata["part_sampled"]

print(f"{'part_sampled':>40} - {'number of pairs':>6}")
for t, count in counts_part_sampled.most_common():
    print(f"{t:>40} - {count:>6,}")

                            part_sampled - number of pairs
                       Multiple Elements - 73,869
                         Vocals / Lyrics - 60,385
                                   Drums - 18,495
                             Hook / Riff - 17,012
                                Dialogue -  4,508
                           Direct Sample -  4,206
                                    Bass -  1,641
                                   Score -    153
         Interpolation (Replayed Sample) -    121
                        Sound FX / Other -    110


In [12]:
# Filter edges with part-sampled 'Dialogue'
for edge_id in list(edges.keys()):
    if edges[edge_id]['part_sampled'] in {'Dialogue'}:
        del edges[edge_id]
print(f"{len(edges):,}")

175,992


In [13]:
# Create a mapping between sources and destinations
src_dst_to_id_mappings = {}
for edge_id, edge in edges.items():
    src_id = edge["src_audio_id"]
    dst_id = edge["dst_audio_id"]
    src_dst_to_id_mappings[(src_id, dst_id)] = edge_id

# Keep only pairs where both audio files were downloaded
paths_wav = set(paths_wav)

edge_ids_downloaded = set()
src_to_dsts = defaultdict(set)  # src_id -> set of dst_ids
dst_to_srcs = defaultdict(set)  # dst_id -> set of src_ids
for annot in edges.values():
    src_id = annot["src_audio_id"]
    dst_id = annot["dst_audio_id"]

    src_path = dir_wav / src_id[:2] / f"{src_id}.wav"
    dst_path = dir_wav / dst_id[:2] / f"{dst_id}.wav"

    if src_path in paths_wav and dst_path in paths_wav:
        src_to_dsts[src_id].add(dst_id)
        dst_to_srcs[dst_id].add(src_id)
        edge_ids_downloaded.add(annot["url"].split("https://www.whosampled.com/sample/")[1].split("/")[0])

print(f"{len(src_to_dsts):>7,} source tracks")
print(f"{len(dst_to_srcs):>7,} destination tracks")
print(f"{len(set(dst_to_srcs.keys()) & set(src_to_dsts.keys())):>7,} both source and destination tracks")
print(f"{len(set(dst_to_srcs.keys()) | set(src_to_dsts.keys())):>7,} unique tracks")

 60,964 source tracks
117,620 destination tracks
  8,244 both source and destination tracks
170,340 unique tracks


## Build the Graph

Source (Sample) to Destination (Sampler) Directed Graph

In [14]:
G = nx.DiGraph()
for src, dsts in src_to_dsts.items():
    for dst in dsts:
        try:
            G.add_edge(src, dst, edge_id=src_dst_to_id_mappings[(src,dst)])
        except Exception as e:
            # These are due to the ID collisions 
            print(e)
print("Has cycles:", not nx.is_directed_acyclic_graph(G))

Has cycles: True


### Remove Simple Cycles

In [15]:
# Find all simple cycles
cycles = list(nx.simple_cycles(G))
print(f"{len(cycles):,} simple cycles found.")
for c in cycles:
    print(c)

self_loops = list(nx.selfloop_edges(G))
print(f"Removing {len(self_loops)} self-loops.")
G.remove_edges_from(self_loops)
# G.remove_nodes_from(list(nx.isolates(G))) # Forgot this...
print("Has cycles:", not nx.is_directed_acyclic_graph(G))

# Remove the cycles from the annotations too
for u, v in self_loops:
    src_to_dsts[u].discard(v)
    dst_to_srcs[v].discard(u)

6 simple cycles found.
['iOKMWSR2Aio']
['YP8JDBOTFfc']
['LwO8IHCi3vw']
['3BT-puAQsgs']
['4xwBT6HwrGk']
['Gb4Tmpu0iXQ']
Removing 6 self-loops.
Has cycles: False


### Display Stats

In [16]:
roots = {n for n in G.nodes if G.in_degree(n) == 0 and G.out_degree(n) > 0}
print(f"{len(roots):,} roots found")

leaves = {n for n in G.nodes if G.out_degree(n) == 0 and G.in_degree(n) > 0}
print(f"{len(leaves):,} leaves found")

52,722 roots found
109,376 leaves found


In [17]:
top_samplers = sorted(G.in_degree(), key=lambda x: x[1], reverse=True)
for track_id, num_samples_used in top_samplers[:20]:
    print(f"{track_id} samples {num_samples_used:>3} tracks.")

fU_s106OHNE samples 106 tracks.
N6bxgWu3xvY samples 102 tracks.
zM5Zx43Sy_c samples 100 tracks.
9FIH6AaV1ss samples  94 tracks.
CsuoIn_JATI samples  86 tracks.
qXo_7pdPGyo samples  84 tracks.
82uvVndso2A samples  84 tracks.
SXTkBpDFR2Q samples  81 tracks.
fNx_3hNhCho samples  74 tracks.
Ww8hRuXQtcA samples  72 tracks.
YQn_oFwkFLQ samples  68 tracks.
JRRzpwwf3qM samples  55 tracks.
BChzoI1W8jA samples  55 tracks.
t_E4NKAifa0 samples  51 tracks.
77-T2HMVKlo samples  49 tracks.
zMxU1OWxboc samples  48 tracks.
YkOOsKFFT7Q samples  47 tracks.
1V8oKMM3m-A samples  44 tracks.
ECnVcpxq0EM samples  44 tracks.
k9KQs-JsA1c samples  43 tracks.


In [18]:
top_sampled = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)
for track_id, num_samplers in top_sampled[:20]:
    print(f"{track_id} is sampled by {num_samplers:>4} tracks.")

GxZuq57_bYM is sampled by 1975 tracks.
Jiqxkdhzi_M is sampled by 1406 tracks.
ZqauvQkC6VQ is sampled by 1250 tracks.
dNP8tbDMZNE is sampled by  823 tracks.
UuR-kDVMoqE is sampled by  484 tracks.
qy8Y7MGGhO4 is sampled by  483 tracks.
K7doq5HouVQ is sampled by  428 tracks.
mZF4G79OLkk is sampled by  426 tracks.
y3RaaHfzcD0 is sampled by  420 tracks.
vLiHBn4G6qg is sampled by  389 tracks.
xRtZ0GFuFG8 is sampled by  378 tracks.
lU4bf3rMqPQ is sampled by  354 tracks.
tUIW9Cwwo6A is sampled by  311 tracks.
wqbEsS5kFb8 is sampled by  310 tracks.
ab7-bVgPkAs is sampled by  303 tracks.
51837yh4hec is sampled by  269 tracks.
IiqO3fB4I78 is sampled by  246 tracks.
V4QuH_eFU4Y is sampled by  242 tracks.
md9veYbl7wI is sampled by  236 tracks.
Ove38w3ztG4 is sampled by  224 tracks.


In [19]:
top_roots = sorted(
    ((node, len(nx.descendants(G, node))) for node in roots),
    key=lambda x: x[1],
    reverse=True
)

# NOTE: family size is #descendants + 1
for track_id, num_descendants in top_roots[:20]:
    print(f"{track_id} has {num_descendants:>4} descendants.")

ZqauvQkC6VQ has 2591 descendants.
GxZuq57_bYM has 2244 descendants.
Jiqxkdhzi_M has 2035 descendants.
dNP8tbDMZNE has 1541 descendants.
UuR-kDVMoqE has 1135 descendants.
y3RaaHfzcD0 has  985 descendants.
K7doq5HouVQ has  945 descendants.
tAnojTvyc0g has  841 descendants.
qy8Y7MGGhO4 has  836 descendants.
mvsYRAc-BWA has  836 descendants.
0JkJCPZA62s has  803 descendants.
4MFQ7JL318o has  705 descendants.
OQLKgyYrfH4 has  625 descendants.
oB0NM6reiRE has  601 descendants.
Ove38w3ztG4 has  599 descendants.
tUIW9Cwwo6A has  576 descendants.
DWKcJwuZnzE has  570 descendants.
b0lQg4iWuHE has  564 descendants.
xRtZ0GFuFG8 has  519 descendants.
mZF4G79OLkk has  494 descendants.


In [20]:
# Get the longest chain
longest_path = nx.dag_longest_path(G)
print(f"Longest chain length: {len(longest_path) - 1} edges / {len(longest_path)} nodes")         
print_chain(longest_path, G)

Longest chain length: 10 edges / 11 nodes

Edge 0  [Vocals / Lyrics]  edge_id=272158
  src: ['Power Records'] — Stacked Cards (Conclusion)  (I-S5AadRE_c)  @ 191s
  dst: ['Ghetto Boys'] — Mind of a Lunatic  (elBAoINk1j8)  @ 0s
  part_sampled: Vocals / Lyrics



Edge 1  [Vocals / Lyrics]  edge_id=179800
  src: ['Ghetto Boys'] — Mind of a Lunatic  (elBAoINk1j8)  @ 185s
  dst: ['Ganksta N-I-P'] — Psycho  (xrIbsaM41lY)  @ 75s
  part_sampled: Vocals / Lyrics



Edge 2  [Vocals / Lyrics]  edge_id=168231
  src: ['Ganksta N-I-P'] — Psycho  (xrIbsaM41lY)  @ 13s
  dst: ['Ganksta N-I-P'] — F**k You  (gpvYMlpnxpM)  @ 3s
  part_sampled: Vocals / Lyrics



Edge 3  [Multiple Elements]  edge_id=640879
  src: ['Ganksta N-I-P'] — F**k You  (gpvYMlpnxpM)  @ 96s
  dst: ['DJ Paul', 'Lord Infamous'] — 187 Invitation  (-0GwMh3iUGg)  @ 38s
  part_sampled: Multiple Elements



Edge 4  [Vocals / Lyrics]  edge_id=303851
  src: ['DJ Paul', 'Lord Infamous'] — 187 Invitation  (-0GwMh3iUGg)  @ 3s
  dst: ['Lord Infamous'] — South Memphis  (AMHkCL66vcc)  @ 42s
  part_sampled: Vocals / Lyrics



Edge 5  [Vocals / Lyrics]  edge_id=303698
  src: ['Lord Infamous'] — South Memphis  (AMHkCL66vcc)  @ 30s
  dst: ['DJ Paul'] — Killaz Off South Parkway  (hSjmnW7DbAg)  @ 0s
  part_sampled: Vocals / Lyrics



Edge 6  [Vocals / Lyrics]  edge_id=263386
  src: ['DJ Paul'] — Killaz Off South Parkway  (hSjmnW7DbAg)  @ 0s
  dst: ['Gangsta Blac'] — Victim of This Shit  (gxlWuertTQw)  @ 0s
  part_sampled: Vocals / Lyrics



Edge 7  [Vocals / Lyrics]  edge_id=564760
  src: ['Gangsta Blac'] — Victim of This Shit  (gxlWuertTQw)  @ 128s
  dst: ['DJ Paul', 'Kingpin Skinny Pimp'] — On My Way Down/Kickin' in Da Door  (V8YZovdlrjo)  @ 110s
  part_sampled: Vocals / Lyrics



Edge 8  [Vocals / Lyrics]  edge_id=558487
  src: ['DJ Paul', 'Kingpin Skinny Pimp'] — On My Way Down/Kickin' in Da Door  (V8YZovdlrjo)  @ 59s
  dst: ['Koopsta Knicca'] — Robbers  (hDxdFvs9H9o)  @ 0s
  part_sampled: Vocals / Lyrics



Edge 9  [Multiple Elements]  edge_id=397936
  src: ['Koopsta Knicca'] — Robbers  (hDxdFvs9H9o)  @ 0s
  dst: ['$uicideboy$', 'RVMIRXZ', 'Black Smurf'] — Loot  (ujcwp-rGZaA)  @ 0s
  part_sampled: Multiple Elements


## Analyze the Weakly Connected Components

In [21]:
def classify_component(G, nodes):
    sub = G.subgraph(nodes)
    n = sub.number_of_nodes()
    e = sub.number_of_edges()
    max_in = max(dict(sub.in_degree()).values())
    max_out = max(dict(sub.out_degree()).values())
    if n == 2 and e == 1:
        return "isolated_pair"
    if e == n - 1 and max_in <= 1 and max_out <= 1:
        return "isolated_chain"
    if e != n - 1:
        return "complex"
    return "tree"

In [22]:
components = list(nx.weakly_connected_components(G))
print(f"{len(components):,} weakly connected components.")

30,053 weakly connected components.


In [23]:
type_counts = Counter()
component_types = {}
for i, comp in enumerate(components):
    t = classify_component(G, comp)
    type_counts[t] += 1
    component_types[i] = (t, comp)

c, n, e = 0, 0, 0
for t, count in type_counts.most_common():
    total_nodes = sum(len(comp) for ct, comp in component_types.values() if ct == t)
    total_edges = sum(G.subgraph(comp).number_of_edges() for ct, comp in component_types.values() if ct == t)
    c += count
    n += total_nodes
    e += total_edges
    print(f"{t:>15}    {count:>6,} components   {total_nodes:>6,} nodes   {total_edges:>7,} edges")
print(f"\nTotal components: {c:,}  nodes: {n:,}  edges: {e:,}")
    
type_mappings = {
    "isolated_pair": [],
    "isolated_chain": [],
    "tree": [],
    "complex": [],
}
for idx, (_type, _) in component_types.items():
    type_mappings[_type].append(idx)

  isolated_pair    22,654 components   45,308 nodes    22,654 edges
           tree     6,964 components   28,883 nodes    21,919 edges
 isolated_chain       397 components    1,195 nodes       798 edges
        complex        38 components   94,954 nodes   127,887 edges

Total components: 30,053  nodes: 170,340  edges: 173,258


In [24]:
print(f"{'component idx':>15} | {'#nodes':>8} | {'#edges':>8}")
print("-" * 37)
for idx in sorted(type_mappings["complex"], key=lambda idx: len(component_types[idx][1]), reverse=True)[:20]:
    nodes = component_types[idx][1]
    edge_count = G.subgraph(nodes).number_of_edges()
    print(f"{idx:>15} | {len(nodes):>8,} | {edge_count:>8,}")

  component idx |   #nodes |   #edges
-------------------------------------
              0 |   94,609 |  127,530
            762 |       48 |       49
           2266 |       30 |       32
            106 |       29 |       30
           1840 |       21 |       26
            701 |       17 |       17
           6082 |       16 |       16
            957 |       12 |       12
            259 |       11 |       11
           1165 |       11 |       11
           3220 |       10 |       10
            322 |        9 |        9
            660 |        9 |        9
           2771 |        9 |        9
           9833 |        9 |       10
           3928 |        8 |        8
           5762 |        7 |        7
            263 |        6 |        6
            896 |        6 |        6
            931 |        6 |        7


## Trim the Mega-component

*NOTE*: This is one of way of trimming the mega-component. Maybe you can do a better job than me...

In [25]:
mega_component = max(components, key=len)

G_clean = G.copy()

# 1 - Find the mega component nodes that have multiple parents
H = G_clean.subgraph(mega_component)
to_remove = [n for n in H if H.in_degree(n) > 1]
print(f"{len(to_remove):,} multi-parent nodes")

# 2 - Remove them from the copy
G_clean.remove_nodes_from(to_remove)

# 3 - Drop singletons stranded within the trimmed component
stranded = [n for n in mega_component if n in G_clean and G_clean.degree(n) == 0]
G_clean.remove_nodes_from(stranded)
print(f"{len(stranded):,} singleton nodes to remove")

# 4 - Verify
trimmed_region = [n for n in mega_component if n in G_clean]
print("Complex component is now a forest:",
      nx.is_forest(G_clean.subgraph(trimmed_region).to_undirected()))

25,050 multi-parent nodes
13,747 singleton nodes to remove
Complex component is now a forest: True


In [26]:
components_clean = list(nx.weakly_connected_components(G_clean))
print(f"{len(components_clean):,} weakly connected components.")

39,625 weakly connected components.


In [27]:
type_counts_clean = Counter()
component_types_clean = {}
for i, comp in enumerate(components_clean):
    t = classify_component(G_clean, comp)
    type_counts_clean[t] += 1
    component_types_clean[i] = (t, comp)

c, n, e = 0, 0, 0
for t, count in type_counts_clean.most_common():
    total_nodes = sum(len(comp) for ct, comp in component_types_clean.values() if ct == t)
    total_edges = sum(
        G_clean.subgraph(comp).number_of_edges() for ct, comp in component_types_clean.values() if ct == t
    )
    c += count
    n += total_nodes
    e += total_edges
    print(f"{t:>15}    {count:>6,} components, {total_nodes:>6,} nodes, {total_edges:>7,} edges.")
print(f"\nTotal components: {c:,}  nodes: {n:,}  edges: {e:,}")
    
type_mappings_clean = {
    "isolated_pair": [],
    "isolated_chain": [],
    "tree": [],
    "complex": [],
}
for idx, (_type, _) in component_types_clean.items():
    type_mappings_clean[_type].append(idx)

  isolated_pair    26,186 components, 52,372 nodes,  26,186 edges.
           tree    12,813 components, 77,047 nodes,  64,234 edges.
 isolated_chain       589 components,  1,779 nodes,   1,190 edges.
        complex        37 components,    345 nodes,     357 edges.

Total components: 39,625  nodes: 131,543  edges: 91,967


In [28]:
print(f"{'component idx':>15} | {'#nodes':>8} | {'#edges':>8}")
print("-" * 37)
for idx in sorted(
    type_mappings_clean["complex"], key=lambda idx: len(component_types_clean[idx][1]), reverse=True
)[:20]:
    nodes = component_types_clean[idx][1]
    edge_count = G_clean.subgraph(nodes).number_of_edges()
    print(f"{idx:>15} | {len(nodes):>8,} | {edge_count:>8,}")

  component idx |   #nodes |   #edges
-------------------------------------
           2148 |       48 |       49
           5176 |       30 |       32
            503 |       29 |       30
           4361 |       21 |       26
           2007 |       17 |       17
          11411 |       16 |       16
           2572 |       12 |       12
            873 |       11 |       11
           2985 |       11 |       11
           6812 |       10 |       10
           1065 |        9 |        9
           1887 |        9 |        9
           6047 |        9 |        9
          16710 |        9 |       10
           7978 |        8 |        8
          10907 |        7 |        7
            887 |        6 |        6
           2441 |        6 |        6
           2513 |        6 |        7
           1329 |        5 |        6


## Train, Validation, Test Splits

### Check Overlap with Sample100 and SamplePairs

Check whether either of the two nodes of an edge exists in the our subset.

In [29]:
yt_ids = set(G_clean.nodes())
print(len(yt_ids))

131543


In [30]:
def extract_yt_id(url):
    if pd.isna(url) or "watch?v=" not in str(url):
        return None
    return str(url).split("watch?v=")[1].split("&")[0]

def get_component_idx(id_node, components):
    # Find the component containing the node
    component_idx = None
    for i, nodes in enumerate(components):
        if id_node in nodes:
            component_idx = i
    if component_idx is None:
        print('Err')
    return component_idx

In [31]:
df_sample100_original_yt = pd.read_csv(dir_sample100 / 'Sample100-whosampled-original-urls.tsv', sep='\t')
sample100_original_yt_ids = set(
    df_sample100_original_yt[["destination_url", "source_url"]]
    .map(extract_yt_id)
    .stack()
    .dropna()
)
print(len(sample100_original_yt_ids))
overlap_yt_ids_sample100 = sample100_original_yt_ids.intersection(yt_ids)
print(len(overlap_yt_ids_sample100))

path_csv_samplepairs = dir_samplepairs / 'meta' / 'samples.csv'
df_samplepairs = pd.read_csv(path_csv_samplepairs, delimiter=',')
samplepairs_yt_ids = {
    row[1]['YTLink1'].split('watch?v=')[1] for row in df_samplepairs.iterrows()
} | {
    row[1]['YTLink2'].split('watch?v=')[1] for row in df_samplepairs.iterrows()
}
print(len(samplepairs_yt_ids))

overlap_yt_ids_samplepairs = samplepairs_yt_ids.intersection(yt_ids)
print(len(overlap_yt_ids_samplepairs))

# Combine overlap IDs from both external datasets
overlap_yt_ids_both = overlap_yt_ids_sample100 | overlap_yt_ids_samplepairs
print(f"{len(overlap_yt_ids_both)} YouTube IDs overlap in total.")

136
96
204
103
199 YouTube IDs overlap in total.


In [32]:
affected_components = defaultdict(list)
for yt_id in overlap_yt_ids_both:
    affected_components[
        get_component_idx(yt_id, components_clean)
    ].append(yt_id)
affected_components = dict(affected_components)

nodes_lost, edges_lost, hits_total = 0, 0, 0
by_type = {}
for comp_idx, hits in affected_components.items():
    comp_type, nodes = component_types_clean[comp_idx]
    # Count the affected elements over the trimmed graph
    edge_count = G_clean.subgraph(nodes).number_of_edges()
    
    if comp_type not in by_type:
        by_type[comp_type] = {"components": 0, "nodes": 0, "edges": 0}
    
    by_type[comp_type]["components"] += 1
    by_type[comp_type]["nodes"] += len(nodes)
    by_type[comp_type]["edges"] += edge_count
    
    nodes_lost += len(nodes)
    edges_lost += edge_count
    hits_total += len(hits)

print(f"{len(affected_components):,} affected components")
print(f"{hits_total:,} overlapping tracks")
print(f"{max([len(v) for v in affected_components.values()])} / {hits_total} belong to the same component.")
print(f"{nodes_lost:,} total nodes in affected components")
print(f"{edges_lost:,} total edges in affected components\n")
print()
print(f"{'comp_type':>16} {'components':>12} {'nodes':>10} {'edges':>10}")
for ct, stats in sorted(by_type.items(), key=lambda x: x[1]["components"], reverse=True):
    print(f"{ct:>16} {stats['components']:>12,} {stats['nodes']:>10,} {stats['edges']:>10,}")

184 affected components
199 overlapping tracks
2 / 199 belong to the same component.
4,576 total nodes in affected components
4,392 total edges in affected components


       comp_type   components      nodes      edges
            tree          146      4,496      4,350
   isolated_pair           34         68         34
  isolated_chain            4         12          8


### Partition

In [33]:
def get_split_edges(restructured_split, G, components):
    """Given a split dict (e.g. restructured['train']), return a flat dict of edge_id -> edge."""
    split_edges = {}
    for comp_type, comp_indices in restructured_split.items():
        for comp_idx in comp_indices:
            nodes = components[comp_idx]
            sub = G.subgraph(nodes)
            for src, dst in sub.edges():
                edge_id = src_dst_to_id_mappings[(src, dst)]
                split_edges[edge_id] = {
                    **edges[edge_id],
                    "edge_id": edge_id,
                    "component_type": comp_type,
                    "component_idx": comp_idx,
                }
    return split_edges

In [34]:
affected_idxs = set(affected_components.keys())
total_comps = len(components_clean)
target = {
    'train': round(0.90 * total_comps),
    'val':   round(0.05 * total_comps),
    'test':  round(0.05 * total_comps),
}

restructured = {'train': {}, 'val': {}, 'test': {}}
filled = {'train': 0, 'val': 0, 'test': 0}

def assign(idx, split):
    ctype = component_types_clean[idx][0]
    restructured[split].setdefault(ctype, []).append(idx)
    filled[split] += 1

# Constrained components go in first and count toward the quotas
for idx in affected_idxs: # external overlap to test
    assign(idx, 'test')
for idx in type_mappings_clean['complex']: # not safely splittable to train
    if idx not in affected_idxs:
        assign(idx, 'train')

# Distribute the rest: fill the small buckets first, train soaks up the remainder.
# Shuffle for an unbiased, reproducible split.
flexible = [
    i for ct in ['isolated_pair', 'tree', 'isolated_chain']
      for i in type_mappings_clean[ct] if i not in affected_idxs
]
random.Random(
    27 # License plate code of Gaziantep, gastronomical capital of Türkiye
).shuffle(flexible)

fill_order = ['test', 'val', 'train']
for idx in flexible:
    split = next((s for s in fill_order if filled[s] < target[s]), 'train')
    assign(idx, split)

for s in ('train', 'val', 'test'):
    print(f"{s:>5}: {filled[s]:>6,} comps  ({filled[s] / total_comps:5.1%})")

train: 35,663 comps  (90.0%)
  val:  1,981 comps  ( 5.0%)
 test:  1,981 comps  ( 5.0%)


In [35]:
comp_indices_train = {idx for indices in restructured["train"].values() for idx in indices}
print(len(comp_indices_train))

comp_indices_val = {idx for indices in restructured["val"].values() for idx in indices}
print(len(comp_indices_val))

comp_indices_test = {idx for indices in restructured["test"].values() for idx in indices}
print(len(comp_indices_test))

print(comp_indices_val.intersection(comp_indices_train))
print(comp_indices_val.intersection(comp_indices_test))
print(comp_indices_train.intersection(comp_indices_test))

35663
1981
1981
set()
set()
set()


In [36]:
singletons = {n for n in G_clean.nodes if G_clean.degree(n) == 0}
print(f"{len(singletons)} edge-less nodes (self-loop leftovers), not exported")

4 edge-less nodes (self-loop leftovers), not exported


In [37]:
train_edges = get_split_edges(restructured['train'], G_clean, components_clean)
val_edges = get_split_edges(restructured['val'], G_clean, components_clean)
test_edges = get_split_edges(restructured['test'], G_clean, components_clean)

split_edges_map = {'train': train_edges, 'val': val_edges, 'test': test_edges}
print(f"{'split':>6} {'components':>12} {'nodes':>10} {'edges':>10}")
for split, edges_dict in split_edges_map.items():
    comp_idxs = {e['component_idx'] for e in edges_dict.values()}
    nodes = set()
    edge_count = 0
    for comp_idx in comp_idxs:
        _, comp_nodes = component_types_clean[comp_idx]
        nodes.update(comp_nodes)
        edge_count += G_clean.subgraph(comp_nodes).number_of_edges()
    print(f"{split:>6} {len(comp_idxs):>12,} {len(nodes):>10,} {edge_count:>10,}")

 split   components      nodes      edges
 train       35,659    114,721     79,111
   val        1,981      6,374      4,393
  test        1,981     10,444      8,463


In [38]:
# Verify the split is a clean partition of G_clean

# Component indices per split (source of truth = restructured)
idx = {s: {i for v in restructured[s].values() for i in v} for s in ('train', 'val', 'test')}

# 1) Mutually exclusive at the component level
assert idx['train'].isdisjoint(idx['val'])
assert idx['train'].isdisjoint(idx['test'])
assert idx['val'].isdisjoint(idx['test'])

# 2) Cover every component — no component dropped or duplicated
assert idx['train'] | idx['val'] | idx['test'] == set(range(len(components_clean)))

# 3) Node level: this is what actually guarantees no track leaks across splits
nodes = {s: set().union(*(components_clean[i] for i in idx[s])) for s in idx}
assert nodes['train'].isdisjoint(nodes['val'])
assert nodes['train'].isdisjoint(nodes['test'])
assert nodes['val'].isdisjoint(nodes['test'])
assert nodes['train'] | nodes['val'] | nodes['test'] == set(G_clean.nodes())

# 4) No Sample100 / SamplePairs track leaks into train or val
assert nodes['train'].isdisjoint(overlap_yt_ids_both)
assert nodes['val'].isdisjoint(overlap_yt_ids_both)

print("partition OK")
for s in ('train', 'val', 'test'):
    print(f"{s:>5}: {len(idx[s]):>6,} comps  {len(nodes[s]):>7,} nodes")
total = sum(len(nodes[s]) for s in nodes)
print(f"total: {total:>7,} nodes  (G_clean: {G_clean.number_of_nodes():,})")

partition OK
train: 35,663 comps  114,725 nodes
  val:  1,981 comps    6,374 nodes
 test:  1,981 comps   10,444 nodes
total: 131,543 nodes  (G_clean: 131,543)


The 4 comp 4 node difference is due to not removing some singleton cycles above.
Realized this while cleaning the notebook. Cannot go back and clean it now as it would change the random partition state.

### Write the splits

In [39]:
# split_dir.mkdir(exist_ok=True, parents=True)

# with open(split_dir / "train.json", "w") as out_f:
#     json.dump(train_edges, out_f)
    
# with open(split_dir / "test.json", "w") as out_f:
#     json.dump(test_edges, out_f)
    
# with open(split_dir / "val.json", "w") as out_f:
#     json.dump(val_edges, out_f)

# audio_ids = set()
# for edge in train_edges.values():
#     audio_ids.add(edge["src_audio_id"])
#     audio_ids.add(edge["dst_audio_id"])
# with open(split_dir / "audio-paths-train.txt", 'w') as out_f:
#     for audio_id in audio_ids:
#         audio_path = dir_wav / audio_id[:2] / f"{audio_id}.wav"
#         out_f.write(f"{str(audio_path)}\n")
        
# audio_ids = set()
# for edge in val_edges.values():
#     audio_ids.add(edge["src_audio_id"])
#     audio_ids.add(edge["dst_audio_id"])
# with open(split_dir / "audio-paths-val.txt", 'w') as out_f:
#     for audio_id in audio_ids:
#         audio_path = dir_wav / audio_id[:2] / f"{audio_id}.wav"
#         out_f.write(f"{str(audio_path)}\n")
        
# audio_ids = set()
# for edge in test_edges.values():
#     audio_ids.add(edge["src_audio_id"])
#     audio_ids.add(edge["dst_audio_id"])

# with open(split_dir / "audio-paths-test.txt", 'w') as out_f:
#     for audio_id in audio_ids:
#         audio_path = dir_wav / audio_id[:2] / f"{audio_id}.wav"
#         out_f.write(f"{str(audio_path)}\n")

In [ ]:
# with open(split_dir / "train.json") as f_in:
#     train_edges = json.load(f_in)
    
# with open(split_dir / "test.json") as f_in:
#     test_edges = json.load(f_in)
    
# with open(split_dir / "val.json") as f_in:
#     val_edges = json.load(f_in)

## Naive Partitioning

What if we naively split the edges? How much node overlap would happen?

In [40]:
def nodes_of(edge_list):
    return {n for e in edge_list for n in e}

records = []
# Do this 10 times
for seed in range(27, 127):
    tr, tmp = train_test_split(list(G.edges()), test_size=0.20, random_state=seed, shuffle=True)
    va, te  = train_test_split(tmp,   test_size=0.50, random_state=seed, shuffle=True)

    n_tr, n_va, n_te = nodes_of(tr), nodes_of(va), nodes_of(te)

    all_nodes = n_tr | n_va | n_te
    multi = sum(
        1 for n in all_nodes
        if (n in n_tr) + (n in n_va) + (n in n_te) > 1
    )

    records.append({
        "n_train": len(n_tr), "n_val": len(n_va), "n_test": len(n_te),
        "train∩val":  len(n_tr & n_va),
        "train∩test": len(n_tr & n_te),
        "val∩test":   len(n_va & n_te),
        "multi_split_nodes": multi,
        "train_val_leak_frac": len(n_te & (n_tr | n_va)) / len(n_te),
        "train_leak_frac": len(n_te & n_tr) / len(n_te),
    })

keys = records[0].keys()
for k in keys:
    v = np.array([r[k] for r in records], dtype=float)
    if "frac" in k:
        print(f"{k:>20}: {v.mean():.4f} ± {v.std():.4f}")
    else:
        print(f"{k:>20}: {v.mean():>10,.1f} ± {v.std():,.1f}")

             n_train:  145,061.3 ± 107.8
               n_val:   26,903.0 ± 72.0
              n_test:   26,914.3 ± 67.9
           train∩val:   13,982.2 ± 74.0
          train∩test:   13,989.8 ± 77.6
            val∩test:    5,035.1 ± 48.6
   multi_split_nodes:   24,078.1 ± 87.6
 train_val_leak_frac: 0.5410 ± 0.0026
     train_leak_frac: 0.5198 ± 0.0027
